# 07b — Skalierungs-Evaluation (Phase 3b)

Vollauswertung auf **n≈200 Instanzen × N Generationen** (seeded, stratifiziert).
Bewusst getrennt vom n=20-**Validitäts**-Notebook (NB 07), das unangetastet bleibt
(alle v1/v2/v3/v4/v5-Caches + Inter-Judge-Agreement gelten weiter).

**Was hier läuft (Kostenentscheidung Phase 3b):**
- Generierung 00/04/05/06 erfolgt in NB 00/04/05/06 (`SCALE_RUN=True`).
- Gewertet wird **nur** der finale **Opus-Judge** (Batch, k=1) + **Cross-Vendor
  gpt-4o-mini** (OpenAI-Batch). v1–v5 + α/τ bleiben das n=20-Validitätsargument.
- Statistik: Bootstrap-CIs, **Varianz innerhalb vs. zwischen Pipelines** (3
  Generationen), paarweise Wilcoxon + Cliff's δ (über Generationen gemittelt),
  Ranking-Stabilität Opus ↔ Cross-Vendor.

**Voraussetzungen:** NB 00/03/04/05/06 mit `SCALE_RUN=True` durchgelaufen;
`ANTHROPIC_API_KEY` und `OPENAI_API_KEY` gesetzt.

In [ ]:
from __future__ import annotations

import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

from utils import (
    scale_instance_ids, N_GENERATIONS_SCALE, RESULTS_DIR, PROMPTS_DIR,
)
from utils.eval import load_scale_records, build_judge_prompt, PIPELINE_LABELS
from utils.judge import judge_batch_sc, parse_judge_response
from utils.batch import make_custom_id
from utils.batch_openai import (
    build_chat_body, chat_request, run_batch as run_openai_batch,
)
from utils.llm import DEFAULT_MODEL, OPENAI_JUDGE_MODEL_TEST, MAX_TOKENS_JUDGE
from utils.stats import bootstrap_ci, wilcoxon_pairwise, cliffs_delta, _delta_magnitude

# ── Konfiguration ─────────────────────────────────────────────────────────────
LOSS_KEY        = 'poisson_log'
PIPELINES       = ['00', '04', '05', '06']
XAI_MODELS      = ['xgb', 'ebm']
PIP_ORDER       = [PIPELINE_LABELS[p] for p in PIPELINES]  # Template, JSON→Text, Vision, Tool-Use
PIPELINE_COLORS = {'Template': '#999999', 'JSON→Text': '#4c72b0',
                   'Vision': '#dd8452', 'Tool-Use': '#55a868'}
METRICS         = ['faithfulness', 'clarity', 'completeness']

OPUS_MODEL          = 'claude-opus-4-8'        # finaler Judge (Ranking-Claim)
OPENAI_JUDGE_MODEL  = OPENAI_JUDGE_MODEL_TEST  # gpt-4o-mini (Cross-Vendor)
JUDGE_K_SCALE       = 1                         # k=1: SC aus, Power aus n×Generationen
JUDGE_SYSTEM        = (PROMPTS_DIR / 'judge_system.md').read_text()

INSTANCE_IDS_SCALE = scale_instance_ids()
N_GEN              = N_GENERATIONS_SCALE

print(f'Instanzen:   {len(INSTANCE_IDS_SCALE)} (seeded, stratifiziert)')
print(f'Generationen: 04/05/06 → {N_GEN}, Template → 1 (deterministisch)')
print(f'Judges:      Opus {OPUS_MODEL} (Batch, k={JUDGE_K_SCALE}) + Cross-Vendor {OPENAI_JUDGE_MODEL} (OpenAI-Batch)')

## 1 · Generierungs-Artefakte laden (gen-aware)

In [ ]:
# load_scale_records liest pipeline{p}/{xai}_inst{iid}[_gen{g}].json:
#   Template (00) → 1 Generation (kein _gen-Suffix), 04/05/06 → N Generationen.
df_scale = load_scale_records(
    PIPELINES, XAI_MODELS, INSTANCE_IDS_SCALE, N_GEN,
    results_dir=RESULTS_DIR, loss_key=LOSS_KEY, require_complete=False,
)
print(f'{len(df_scale)} Erklärungen geladen.')
display(df_scale.groupby(['pipeline_label', 'xai_model']).size().unstack())
print('\nGenerationen je Pipeline:')
display(df_scale.groupby('pipeline_label')['generation'].nunique())

## 2 · Finaler Judge — Opus (Batch, k=1)

Ein Batch-Request je Erklärung (alle Generationen). Opus zieht seine Diversität
aus der Default-Stochastik (kein `temperature`-Feld); bei k=1 ist der „Median"
schlicht der Einzelscore. Ergebnis schema-identisch zu NB 07 (`judge_batch_sc`).

In [ ]:
opus_scale_path = RESULTS_DIR / 'eval_llm_judge_opus_scale.json'

def _judge_entries(df, prefix):
    """Baut (base_cid, prompt) je Zeile; Tool-Trace nur für Pipeline 06."""
    entries, meta = [], []
    for _, row in df.iterrows():
        base_cid = make_custom_id(prefix, row['pipeline'], row['xai_model'],
                                  row['instance_id'], f"g{row['generation']}")
        trace = row['tool_calls'] if row['pipeline'] == '06' else None
        prompt = build_judge_prompt(row.to_dict(), row['xai_model'],
                                    int(row['instance_id']), loss_key=LOSS_KEY,
                                    tool_trace=trace)
        entries.append((base_cid, prompt))
        meta.append((base_cid, row['pipeline_label'], row['xai_model'],
                     int(row['instance_id']), int(row['generation'])))
    return entries, meta


def _rows_from_results(meta, lookup):
    """meta + {cid→scores-dict} → Liste von Judge-Zeilen (mit generation)."""
    out = []
    for base_cid, plabel, xai, iid, gen in meta:
        r = lookup.get(base_cid, {}) or {}
        out.append({
            'pipeline_label': plabel, 'xai_model': xai, 'instance_id': iid,
            'generation': gen,
            'faithfulness': r.get('faithfulness'), 'clarity': r.get('clarity'),
            'completeness': r.get('completeness'),
        })
    return out


_opus_cached = json.loads(opus_scale_path.read_text()) if opus_scale_path.exists() else []
if _opus_cached and len(_opus_cached) == len(df_scale):
    print(f'Opus-Scale-Ergebnisse aus Cache: {opus_scale_path.name} ({len(_opus_cached)})')
    judge_opus_df = pd.DataFrame(_opus_cached)
else:
    entries, meta = _judge_entries(df_scale, 'jOp')
    print(f'Reiche {len(entries)} Opus-Judge-Requests als Batch ein (k={JUDGE_K_SCALE}) …')
    sc = judge_batch_sc(
        entries, system=JUDGE_SYSTEM, model=OPUS_MODEL,
        max_tokens=MAX_TOKENS_JUDGE, k=JUDGE_K_SCALE, temperature=None,
        state_path=RESULTS_DIR / '_judge_opus_scale_state.json',
    )
    opus_rows = _rows_from_results(meta, sc)
    opus_scale_path.write_text(json.dumps(opus_rows, indent=2, ensure_ascii=False))
    judge_opus_df = pd.DataFrame(opus_rows)

for c in METRICS:
    judge_opus_df[c] = pd.to_numeric(judge_opus_df[c], errors='coerce')

n_valid = judge_opus_df[METRICS].notna().all(axis=1).sum()
print(f'Opus: {n_valid}/{len(judge_opus_df)} vollständig geparst.')
display(judge_opus_df.groupby('pipeline_label')[METRICS].mean().round(2).reindex(PIP_ORDER))

## 3 · Cross-Vendor-Judge — gpt-4o-mini (OpenAI-Batch)

Identische Rubrik, anderer Anbieter → Selbst-Präferenz-/Ranking-Stabilität bei
n≈200. `temperature=0` (gpt-4o-mini akzeptiert sie → deterministisch als Messinstrument).

In [ ]:
oai_scale_path = RESULTS_DIR / 'eval_llm_judge_openai_scale.json'

_oai_cached = json.loads(oai_scale_path.read_text()) if oai_scale_path.exists() else []
if _oai_cached and len(_oai_cached) == len(df_scale):
    print(f'Cross-Vendor-Scale aus Cache: {oai_scale_path.name} ({len(_oai_cached)})')
    judge_oai_df = pd.DataFrame(_oai_cached)
else:
    _, meta = _judge_entries(df_scale, 'jOa')
    requests = []
    for (base_cid, _plabel, _xai, _iid, _gen), (_, row) in zip(meta, df_scale.iterrows()):
        trace = row['tool_calls'] if row['pipeline'] == '06' else None
        prompt = build_judge_prompt(row.to_dict(), row['xai_model'],
                                    int(row['instance_id']), loss_key=LOSS_KEY,
                                    tool_trace=trace)
        body = build_chat_body(prompt, system=JUDGE_SYSTEM, model=OPENAI_JUDGE_MODEL,
                               max_tokens=MAX_TOKENS_JUDGE, temperature=0.0)
        requests.append(chat_request(base_cid, body))
    print(f'Reiche {len(requests)} Cross-Vendor-Requests als OpenAI-Batch ein …')
    outcome = run_openai_batch(
        requests, parse=parse_judge_response,
        state_path=RESULTS_DIR / '_judge_openai_scale_state.json',
    )
    oai_rows = _rows_from_results(meta, outcome['succeeded'])
    if outcome['failed']:
        print(f"⚠️  {len(outcome['failed'])} Requests endgültig fehlgeschlagen.")
    oai_scale_path.write_text(json.dumps(oai_rows, indent=2, ensure_ascii=False))
    judge_oai_df = pd.DataFrame(oai_rows)

for c in METRICS:
    judge_oai_df[c] = pd.to_numeric(judge_oai_df[c], errors='coerce')

n_valid = judge_oai_df[METRICS].notna().all(axis=1).sum()
print(f'Cross-Vendor: {n_valid}/{len(judge_oai_df)} vollständig geparst.')
display(judge_oai_df.groupby('pipeline_label')[METRICS].mean().round(2).reindex(PIP_ORDER))

## 4 · Statistik — Bootstrap-CI (Opus, voller Lauf)

Jede Generation ist eine Beobachtung. CI über alle Beobachtungen je Pipeline.

In [ ]:
ci_rows = []
for pipeline in PIP_ORDER:
    sub = judge_opus_df[judge_opus_df['pipeline_label'] == pipeline]
    entry = {'pipeline': pipeline, 'n': len(sub)}
    for m in METRICS:
        lo, hi, obs = bootstrap_ci(sub[m].dropna().values)
        entry[f'{m}_mean'] = round(obs, 3)
        entry[f'{m}_ci']   = f'{obs:.2f} [{lo:.2f}, {hi:.2f}]'
    ci_rows.append(entry)
ci_df = pd.DataFrame(ci_rows).set_index('pipeline')
print(f'95 %-Bootstrap-CI — Opus-Judge (n≈{int(ci_df["n"].mean())} Beob./Pipeline)')
display(ci_df[[f'{m}_ci' for m in METRICS]].rename(columns={f'{m}_ci': m.capitalize() for m in METRICS}))
ci_df.to_csv(RESULTS_DIR / 'eval_scale_bootstrap_ci.csv')

## 5 · Varianz innerhalb vs. zwischen Pipelines

**Innerhalb:** Streuung der Scores über die N Generationen derselben Einheit
(Instanz × Modell) — misst die Stochastik der Generierung. Template = 0
(deterministisch). **Zwischen:** Streuung der Pipeline-Mittelwerte.

In [ ]:
var_rows = []
for m in METRICS:
    # innerhalb: Std über Generationen je (pipeline, xai, instance), dann Mittel
    within = (judge_opus_df
              .groupby(['pipeline_label', 'xai_model', 'instance_id'])[m]
              .std(ddof=0))
    within_by_pipe = within.groupby('pipeline_label').mean().reindex(PIP_ORDER)
    # zwischen: Std der Pipeline-Mittelwerte
    pipe_means = judge_opus_df.groupby('pipeline_label')[m].mean().reindex(PIP_ORDER)
    between = float(pipe_means.std(ddof=0))
    for pipeline in PIP_ORDER:
        var_rows.append({
            'metric': m, 'pipeline': pipeline,
            'within_gen_std': round(float(within_by_pipe.get(pipeline, np.nan)), 4),
        })
    print(f'{m:14s}: zwischen-Pipeline-Std = {between:.3f}  |  '
          f'Ø innerhalb-Generationen-Std = {within_by_pipe.mean():.3f}')

var_df = pd.DataFrame(var_rows)
within_tbl = var_df.pivot(index='pipeline', columns='metric', values='within_gen_std').reindex(PIP_ORDER)
print('\nØ Within-Generationen-Std je Pipeline (Generierungs-Stochastik):')
display(within_tbl)
var_df.to_csv(RESULTS_DIR / 'eval_scale_variance.csv', index=False)

## 6 · Paarweise Wilcoxon + Cliff's δ (Opus)

Für den gepaarten Test werden die N Generationen je (Instanz × Modell) zum
Mittelwert aggregiert → ein gepaarter Wert je Einheit über alle Pipelines.

In [ ]:
# Aggregation über Generationen → ein Wert je (pipeline, instance, xai)
df_agg = (judge_opus_df
          .groupby(['pipeline_label', 'xai_model', 'instance_id'], as_index=False)[METRICS]
          .mean())

print('Paarweise Wilcoxon signed-rank (zweiseitig) + Cliff\'s δ — Opus, n≈200 (gen-gemittelt)')
print('Gepaart über (instance_id, xai_model).\n')
wilcoxon_all = []
for m in METRICS:
    wdf = wilcoxon_pairwise(df_agg, PIP_ORDER, m)
    wdf.insert(0, 'metric', m)
    wilcoxon_all.append(wdf)
    print(f'── {m.upper()} ──')
    display(wdf[['pipeline_a', 'pipeline_b', 'n_pairs', 'mean_a', 'mean_b',
                 'delta_mean', 'p_value', 'cliffs_d', 'magnitude']])
    print()
wilcoxon_all_df = pd.concat(wilcoxon_all, ignore_index=True)
wilcoxon_all_df.to_csv(RESULTS_DIR / 'eval_scale_wilcoxon_cliffsdelta.csv', index=False)
print(f'Gespeichert: {RESULTS_DIR / "eval_scale_wilcoxon_cliffsdelta.csv"}')

## 7 · Ranking-Stabilität: Opus ↔ Cross-Vendor

Kernfrage: Bleibt das Pipeline-Ranking stabil, wenn ein anderer Anbieter bewertet?

In [ ]:
from scipy.stats import spearmanr

def _pipe_totals(jdf):
    return jdf.groupby('pipeline_label')[METRICS].mean().sum(axis=1).reindex(PIP_ORDER)

opus_tot = _pipe_totals(judge_opus_df)
oai_tot  = _pipe_totals(judge_oai_df)

rank_df = pd.DataFrame({
    'Opus_Score':       opus_tot.round(2),
    'Opus_Rang':        opus_tot.rank(ascending=False).astype(int),
    'CrossVendor_Score': oai_tot.round(2),
    'CrossVendor_Rang':  oai_tot.rank(ascending=False).astype(int),
}).loc[PIP_ORDER]
print('Gesamtranking (Σ Faithfulness+Clarity+Completeness, max=15):')
display(rank_df)

print('\nSpearman-ρ je Kriterium (Pipeline-Rang, n=4):')
for m in METRICS:
    o = judge_opus_df.groupby('pipeline_label')[m].mean().reindex(PIP_ORDER)
    a = judge_oai_df.groupby('pipeline_label')[m].mean().reindex(PIP_ORDER)
    rho, p = spearmanr(o.rank(), a.rank())
    print(f'  {m:14s}: ρ = {rho:.3f}  (p = {p:.3f})')

rho_tot, _ = spearmanr(opus_tot.rank(), oai_tot.rank())
print(f'\nGesamtrang Opus↔Cross-Vendor: Spearman ρ = {rho_tot:.3f}')
rank_df.to_csv(RESULTS_DIR / 'eval_scale_ranking_stability.csv')
print('Interpretation: ρ ≈ 1 → Ranking anbieterstabil; ρ < 0.6 → Selbst-Präferenz möglich.')

## 8 · Plot — Pipeline-Scores mit CI (Opus, n≈200)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x = np.arange(len(PIP_ORDER))
for ax_i, m in enumerate(METRICS):
    means, los, his = [], [], []
    for pipeline in PIP_ORDER:
        sub = judge_opus_df[judge_opus_df['pipeline_label'] == pipeline][m].dropna().values
        lo, hi, obs = bootstrap_ci(sub)
        means.append(obs); los.append(obs - lo); his.append(hi - obs)
    colors = [PIPELINE_COLORS[p] for p in PIP_ORDER]
    axes[ax_i].bar(x, means, yerr=[los, his], color=colors, capsize=4)
    axes[ax_i].set_xticks(x); axes[ax_i].set_xticklabels(PIP_ORDER, fontsize=9, rotation=10)
    axes[ax_i].set_ylim(0, 5.5); axes[ax_i].set_title(m.capitalize())
    axes[ax_i].axhline(4, color='#aaa', ls='--', lw=0.7, alpha=0.6)
plt.suptitle(f'Phase 3b — Opus-Judge, n≈{len(INSTANCE_IDS_SCALE)} Instanzen × {N_GEN} Gen. (95 %-Bootstrap-CI)', y=1.02)
plt.tight_layout()
out = RESULTS_DIR / 'eval_scale_scores.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); plt.show()
print(f'Gespeichert: {out}')